# Sector-Specific Applicant Name Harmonization (Stage 2 Rectification)

Modular post-processing pipeline that takes the generic Stage 1 harmonization output (`harmonization_lookup-2.csv`) and the raw PATSTAT applicant table (`qc_applicants_PROD.csv`), partitions organizations by entity category (`company`, `university`, `gov / non-profit`), and decomposes oversized clusters using squared-IDF weighting and distinguishing-core guards.

In [5]:
%pip install rapidfuzz sentence-transformers networkx

  Using cached rapidfuzz-3.14.6-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (12 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 114.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.3/780.3 kB 65.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.0/857.0 kB 72.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 132.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.5/13.5 MB 154.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 155.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 809.2/809.2 kB 73.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 539.2/539.2 MB 46.5 MB/s  0:00:07m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.2/519.2 MB 49.0 MB/s  0:00:07m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 70.4 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.0/216.0 MB 69.8 MB/s  0:00:03m0:00

In [6]:
import re
import unicodedata
from collections import Counter

import numpy as np
import pandas as pd
import networkx as nx
import networkx.algorithms.community as nx_comm
from rapidfuzz.fuzz import ratio as rf_ratio
from sentence_transformers import SentenceTransformer

# --- File Paths ---
RAW_APPLICANTS_CSV = "qc_applicants_PROD.csv"
STAGE1_LOOKUP_CSV = "harmonization_lookup_stage1.csv"
OUTPUT_LOOKUP_CSV = "harmonization_lookup.csv"
OUTPUT_FULL_CSV = "harmonization_output.csv"

# --- Model & Rectification Hyperparameters ---
MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
MIN_CLUSTER_SIZE_TO_SPLIT = 6     # only decompose Stage 1 blocks with >= 6 distinct names
SCORE_THRESHOLD = 0.56            # aligned with Stage 1 validated Otsu threshold (~0.559) better double check this later
CORE_JAC_MIN = 0.35               # minimum squared-IDF overlap on distinguishing core tokens
CORE_EDIT_MIN = 0.80              # minimum core character similarity if core overlap < 0.35
LOUVAIN_RESOLUTION = 1.35
LOUVAIN_SEED = 42

## 1. Entity-Type Classification & Filtering
Classify applicants using PATSTAT `psn_sector` first, falling back to multilingual regex patterns when `psn_sector` is missing or `UNKNOWN`[cite: 1]. Drop individuals and `-NOT AVAILABLE-` records so only organizations enter harmonization[cite: 1].

In [7]:
LEGAL = [
    "INC", "LLC", "LLLP", "LL", "LTD", "LIMITED", "CORP", "CORPORATION", "CO", "COMPANY", "COPMANY",
    "GMBH", "AG", "SE", "SA", "SAS", "SARL", "SRL", "SPA", "SL", "BV", "NV", "NA", "OY", "AB", "AS",
    "ASA", "PTY", "PLC", "KK", "LLP", "LP", "PTE", "KG", "OOO", "SP Z OO", "SPOLKA", "JOINT.STOCK",
    "ANONIM SIRKETI", "OBSHCHESTVO"
]
LEGAL_CJK = ["有限公司", "株式会社", "股份", "公司", "会社", "주식회사", "インコーポレイテッド", "アクチボラグ"]
UNIV = [
    "UNIV", "COLLEGE", "POLYTECHN", "POLITECNIC", "ECOLE", "HOCHSCHULE", "SCHOOL",
    "INSTITUTE OF TECHNOLOGY", "GIJUKU", "TRUSTEES", "EDUCATIONAL FUND",
    "大学", "大學", "学院", "カレッジ", "義塾", r"\bETH\b", r"\bEPFL\b", r"\bUESTC\b"
]
GOV = [
    "INSTIT", "INSTYTUT", "INSTUTUC", "ИНСТИТУТ", "ACADEM", "COMMISSARIAT", "CNRS", "LABORATORY",
    "CENTER", "CENTRE", "CENTRUM", "ZENTRUM", "FOUNDATION", "FONDAZIONE", "FUNDACIO", "STIFTUNG",
    "FRAUNHOFER", "FROUNHAV", "PLANCK", "GOVERNMENT", "MINISTRY", "MINISTERIUM", "BUNDES",
    "FEDERATSIYA", "AGENTSTVO", "UCHREZHDENIE", "SECRETARY", "AGENCY", "COUNCIL", "CONSEJO",
    "CONSIGLIO", r"\bNATIONAL\b", "IDRYMA", "ASSOCIATION", "HOSPITAL", "ARMY", "AIR FORCE", "NAVY",
    "RIKEN", r"\bTNO\b", "NASA", "研究", "中心", "委员会", "科学院", "センター"
]
FIRM = [
    "TECHNOLOG", "QUANT", "COMPUT", "SYSTEM", "SOLUTION", r"\bLABS?\b", "SOFTWARE", "NETWORK",
    "ELECTRONIC", "SEMICONDUCT", "PHOTONIC", "DEVICES", "INDUSTR", "HOLDING", "GROUP", "CAPITAL",
    "VENTURES", "ENTERPRISE", "INTERNATIONAL", "SCIENCE", "DIGITAL", "DATA", "INTELLIGEN", "ENERGY",
    "PHARMA", "SECUR", "MACHINES", "INSTRUMENTS", "ENGINEERING", "INNOVATION", "BLOCKCHAIN",
    "BANK", "INSURANCE", "PORTFOLIO", r"\bAI\b", "&", r"\+", "科技", "技术", "量子"
]

LEGAL_RE = re.compile(r"\b(" + "|".join(LEGAL) + r")\b|" + "|".join(LEGAL_CJK))
UNIV_RE, GOV_RE, FIRM_RE = (re.compile("|".join(words)) for words in (UNIV, GOV, FIRM))
WORD = r"[^\W\d_]+(?:['\-][^\W\d_]+)*"
PERSON_COMMA = re.compile(rf"^{WORD}( {WORD}){{0,3}} ?, ?{WORD}")
PERSON_WORDS = re.compile(rf"^{WORD}( {WORD}){{1,3}}$")

def normalize_for_type(name):
    s = unicodedata.normalize("NFKC", str(name))
    s = "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c)).upper()
    s = " ".join(s.replace("?", "").replace(".", "").split())
    return s.replace("UNIVERSITY CORPORATION", "UNIVERSITY")

def classify_entity(psn_name, psn_sector=None):
    s = str(psn_sector).upper() if pd.notna(psn_sector) else ""
    if "UNIVERSITY" in s:             return "university"
    if "GOV" in s or "HOSPITAL" in s: return "gov / non-profit"
    if "COMPANY" in s:                return "company"
    if "INDIVIDUAL" in s:             return "individual"

    n = normalize_for_type(psn_name)
    if LEGAL_RE.search(n): return "company"
    if UNIV_RE.search(n):  return "university"
    if GOV_RE.search(n):   return "gov / non-profit"
    if FIRM_RE.search(n):  return "company"
    if PERSON_COMMA.match(n) or PERSON_WORDS.match(n) or "·" in n:
        return "individual"
    if len(n.split()) == 1: return "company"
    return "uncertain"

# Load and merge Stage 1 lookup with raw PATSTAT sector metadata
raw_df = pd.read_csv(RAW_APPLICANTS_CSV)
raw_meta = raw_df[["psn_name", "psn_sector"]].drop_duplicates("psn_name")
stage1_df = pd.read_csv(STAGE1_LOOKUP_CSV)

df_work = stage1_df.merge(raw_meta, on="psn_name", how="left")
df_work["entity_type"] = [classify_entity(n, sec) for n, sec in zip(df_work["psn_name"], df_work["psn_sector"])]

# Apply targeted manual overrides for edge cases
df_work.loc[df_work["psn_name"].isin([
    "DHITECH - DISTRETTO TECNOLOGICO H IGH TECH S.C.A R.L.",
    "PEARL COHEN ZEDEK LATZER BARATZ"
]), "entity_type"] = "company"
df_work.loc[df_work["psn_name"].str.contains("EHNSEHRM", na=False), "entity_type"] = "gov / non-profit"

df_work = df_work[(df_work["entity_type"] != "individual") & (df_work["psn_name"] != "-NOT AVAILABLE-")].copy()
print("Unique cleaned organizations by category:")
print(df_work.drop_duplicates("clean")["entity_type"].value_counts())

Unique cleaned organizations by category:
entity_type
company             1658
university           741
gov / non-profit     250
uncertain              1
Name: count, dtype: int64


## 2. Corpus Statistics & Semantic Embeddings
Compute squared inverse document frequency weights ($\text{idf}(t)^2$) across the sector corpus and encode unique cleaned organization names once (or reuse cached embeddings if already in memory).

In [8]:
BOILERPLATE_CORE = {
    "UNIV", "INST", "COLLEGE", "ACADEMY", "SCHOOL", "HOCHSCHULE", "ECOLE", "POLITECNICA", "POLYTECHNICAL",
    "TECH", "SCIENCE", "RESEARCH", "LAB", "CENTER", "GROUP", "CORP", "COMPANY", "HOLDINGS", "INTERNATIONAL",
    "NATIONAL", "STATE", "PUBLIC", "ROYAL", "NORMAL", "GENERAL", "CENTRAL", "FEDERAL",
    "KORPERSCHAFT", "OFFENTLICHEN", "RECHTS", "BOARD", "REGENTS", "TRUSTEES", "FOUNDATION", "ASSOCIATION",
    "QUANTUM", "COMPUTING", "COMPUTER", "INFORMATION", "SYSTEMS", "NETWORK", "ELECTRONIC", "ELECTRONICS",
    "ELECTRIC", "POWER", "GRID", "ENERGY", "DIGITAL", "INTELLIGENT", "SMART", "DATA", "SOFTWARE",
    "CHINA", "CHINESE", "BEIJING", "SHANGHAI", "SHENZHEN", "GUANGDONG", "ANHUI", "JIANGSU", "ZHEJIANG",
    "HEFEI", "NANJING", "SUZHOU", "HANGZHOU", "WUHAN", "CHENGDU", "CHONGQING", "DALIAN", "HARBIN",
    "NORTH", "SOUTH", "EAST", "WEST", "NORTHEAST", "NORTHWEST", "SOUTHEAST", "SOUTHWEST", "NEW"
}

unique_clean_names = sorted(df_work["clean"].dropna().unique())
N_CORPUS = len(unique_clean_names)
df_counts = Counter(t for name in unique_clean_names for t in set(name.split()))
idf_sq = {t: (np.log(N_CORPUS / c)) ** 2 for t, c in df_counts.items()}

# Encode unique names (or reuse `names` and `emb` if already present in notebook memory)
if "names" in globals() and "emb" in globals() and set(unique_clean_names).issubset(set(names)):
    name_to_idx = {n: idx for idx, n in enumerate(names)}
    emb_matrix = emb[[name_to_idx[n] for n in unique_clean_names]]
    print(f"Reused cached embeddings for {N_CORPUS:,} unique names.")
else:
    sbert_model = SentenceTransformer(MODEL_NAME)
    emb_matrix = sbert_model.encode(unique_clean_names, normalize_embeddings=True, show_progress_bar=True).astype(np.float32)
    print(f"Encoded {N_CORPUS:,} unique names.")

clean_to_emb_idx = {n: idx for idx, n in enumerate(unique_clean_names)}

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/83 [00:00<?, ?it/s]

Encoded 2,650 unique names.


## 3. Rectification Scoring & Louvain Sub-Clustering
Define `refine_subcluster()`, which decomposes oversized `(harm_id, entity_type)` blocks using squared-IDF Jaccard, distinguishing-core guards, and tightened Louvain community detection.

In [9]:
def refine_subcluster(member_names, th=SCORE_THRESHOLD, res=LOUVAIN_RESOLUTION):
    m = len(member_names)
    if m < MIN_CLUSTER_SIZE_TO_SPLIT:
        return [set(member_names)]

    idxs = [clean_to_emb_idx[n] for n in member_names]
    cos_mat = emb_matrix[idxs] @ emb_matrix[idxs].T

    toks = [set(n.split()) for n in member_names]
    core_toks = [({t for t in tk if t not in BOILERPLATE_CORE} or tk) for tk in toks]
    core_strs = [" ".join(t for t in n.split() if t not in BOILERPLATE_CORE) or n for n in member_names]
    tok_sums = [sum(idf_sq.get(t, 0.0) for t in tk) for tk in toks]
    core_sums = [sum(idf_sq.get(t, 0.0) for t in tk) for tk in core_toks]

    G_sub = nx.Graph()
    G_sub.add_nodes_from(member_names)

    for i in range(m):
        for j in range(i + 1, m):
            cos = float(cos_mat[i, j])

            # Squared-IDF Jaccard on full cleaned tokens
            inter = toks[i] & toks[j]
            if inter:
                inter_w = sum(idf_sq.get(t, 0.0) for t in inter)
                jac = inter_w / max(tok_sums[i] + tok_sums[j] - inter_w, 1e-9)
            else:
                jac = 0.0

            # Distinguishing-core guard (prevents institutional/sector template merges)
            c_inter = core_toks[i] & core_toks[j]
            if c_inter:
                c_inter_w = sum(idf_sq.get(t, 0.0) for t in c_inter)
                c_jac = c_inter_w / max(core_sums[i] + core_sums[j] - c_inter_w, 1e-9)
            else:
                c_jac = 0.0
            c_ed = rf_ratio(core_strs[i], core_strs[j]) / 100.0

            if c_jac < CORE_JAC_MIN and c_ed < CORE_EDIT_MIN:
                continue

            ed = rf_ratio(member_names[i], member_names[j]) / 100.0
            score = (cos + jac + ed) / 3.0
            if score >= th:
                G_sub.add_edge(member_names[i], member_names[j], weight=score)

    return list(nx_comm.louvain_communities(G_sub, weight="weight", resolution=res, seed=LOUVAIN_SEED))

## 4. Execute Stage 2 Partitioning & Canonical Naming
Split every Stage 1 cluster by `entity_type`, refine oversized sub-clusters, and assign deterministic `HARM_XXXXX` IDs and frequency-weighted canonical names.

In [10]:
final_communities = []
for (hid, etype), grp in df_work.groupby(["harm_id", "entity_type"]):
    members = sorted(grp["clean"].dropna().unique())
    for sub_comm in refine_subcluster(members):
        final_communities.append((etype, sub_comm))

# Weight canonical name selection by frequency in the full raw PATSTAT dataset
raw_freq = raw_df.merge(df_work[["psn_name", "clean"]].drop_duplicates(), on="psn_name", how="inner")
clean_weights = raw_freq["clean"].value_counts().to_dict()

final_communities.sort(key=lambda item: (-sum(clean_weights.get(n, 0) for n in item[1]), min(item[1])))

clean_to_id, clean_to_canon, clean_to_type = {}, {}, {}
for idx, (etype, members) in enumerate(final_communities, start=1):
    new_id = f"HARM_{idx:05d}"
    best_name = min(members, key=lambda n: (-clean_weights.get(n, 0), len(n), n))
    for n in members:
        clean_to_id[n] = new_id
        clean_to_canon[n] = best_name
        clean_to_type[n] = etype

df_work["harm_id_final"] = df_work["clean"].map(clean_to_id)
df_work["harmonized_name_final"] = df_work["clean"].map(clean_to_canon)

print(f"Stage 1 clusters: {df_work['harm_id'].nunique():,} -> Stage 2 rectified clusters: {len(final_communities):,}")

Stage 1 clusters: 1,359 -> Stage 2 rectified clusters: 2,040


## 5. Quality Diagnostics & Export
Inspect the largest post-rectification clusters and save both the lookup dictionary (`harmonization_lookup_rectified.csv`) and the full enriched PATSTAT applicant table (`qc_applicants_harmonized_PROD.csv`).

In [11]:
cluster_audit = (
    df_work.groupby(["harm_id_final", "entity_type", "harmonized_name_final"])
    .agg(n_raw=("psn_name", "nunique"), n_clean=("clean", "nunique"))
    .reset_index()
    .sort_values("n_clean", ascending=False)
)

print("Cluster size distribution (n_clean):")
print(cluster_audit["n_clean"].describe(percentiles=[0.5, 0.9, 0.95, 0.99]))
print("\nTop 20 largest harmonized organizations:")
print(cluster_audit.head(20).to_string(index=False))

# Export lookup table
lookup_out = (
    df_work[["psn_name", "clean", "entity_type", "harm_id_final", "harmonized_name_final"]]
    .drop_duplicates()
    .sort_values(["harm_id_final", "psn_name"])
)
lookup_out.to_csv(OUTPUT_LOOKUP_CSV, index=False, encoding="utf-8-sig")

# Merge back onto full PATSTAT applicant table
full_out = raw_df.merge(
    lookup_out[["psn_name", "clean", "entity_type", "harm_id_final", "harmonized_name_final"]].drop_duplicates("psn_name"),
    on="psn_name",
    how="inner"
)
full_out.to_csv(OUTPUT_FULL_CSV, index=False, encoding="utf-8-sig")
print(f"\nSaved lookup ({len(lookup_out):,} rows) -> {OUTPUT_LOOKUP_CSV}")
print(f"Saved full dataset ({len(full_out):,} rows) -> {OUTPUT_FULL_CSV}")

Cluster size distribution (n_clean):
count    2045.000000
mean        1.299756
std         0.792347
min         1.000000
50%         1.000000
90%         2.000000
95%         3.000000
99%         5.000000
max         9.000000
Name: n_clean, dtype: float64

Top 20 largest harmonized organizations:
harm_id_final      entity_type                                                                                                                              harmonized_name_final  n_raw  n_clean
   HARM_00285          company                                                                                                                   STATE GRID FUJIAN ELECTRIC POWER     10        9
   HARM_00196          company                                                                                INFORMATION COMMUNICATION BRANCH STATE GRID ZHEJIANG ELECTRIC POWER      9        9
   HARM_00181          company                                                                                          